In [1]:
!pip install diffusers transformers accelerate scipy torchmetrics pillow torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.3/927.3 kB 21.5 MB/s eta 0:00:00


In [2]:
import torch
from diffusers import StableDiffusionPipeline
from torchmetrics.functional.multimodal import clip_score
from torchvision.transforms import ToTensor
from PIL import Image
import os

# Define the models to compare
MODELS = {
    "Runway Stable Diffusion": "runwayml/stable-diffusion-v1-5",
    "Dreamlike Photoreal 2.0": "dreamlike-art/dreamlike-photoreal-2.0",
}

# Define prompts to evaluate
PROMPTS = [
    "A futuristic city with flying cars and neon lights at night",
    "A serene beach with crystal clear water and a sunset",
    "A hyper-realistic portrait of a smiling old man with wrinkles and gray hair",
]

# Output directory for saving generated images
OUTPUT_DIR = "model_comparison_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Function to evaluate models
def evaluate_models(models, prompts, output_dir):
    results = []

    for model_name, model_id in models.items():
        print(f"Evaluating model: {model_name}")

        # Load the model
        pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
        pipe.to("cuda")

        for idx, prompt in enumerate(prompts):
            print(f"Generating image for prompt: {prompt}")

            # Generate image
            image = pipe(prompt).images[0]

            # Save image
            model_output_dir = os.path.join(output_dir, model_name.replace(" ", "_"))
            os.makedirs(model_output_dir, exist_ok=True)
            file_path = os.path.join(model_output_dir, f"image_{idx + 1}.png")
            image.save(file_path)

            # Convert image to tensor for CLIP score
            image_tensor = ToTensor()(image).unsqueeze(0).to("cuda")

            # Compute CLIP score
            clip_similarity = clip_score(image_tensor, prompt)
            print(f"CLIP Score: {clip_similarity.item()}")

            # Store results
            results.append({
                "model": model_name,
                "prompt": prompt,
                "image_path": file_path,
                "clip_score": clip_similarity.item(),
            })

    return results


# Evaluate the models
results = evaluate_models(MODELS, PROMPTS, OUTPUT_DIR)

# Display the results
for res in results:
    print(f"Model: {res['model']}")
    print(f"Prompt: {res['prompt']}")
    print(f"Image Path: {res['image_path']}")
    print(f"CLIP Score: {res['clip_score']}")
    print()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Evaluating model: Runway Stable Diffusion


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

...

CLIP Score: 15.42233943939209
Generating image for prompt: A serene beach with crystal clear water and a sunset


  0%|          | 0/50 [00:00<?, ?it/s]

CLIP Score: 14.615382194519043
Generating image for prompt: A hyper-realistic portrait of a smiling old man with wrinkles and gray hair


  0%|          | 0/50 [00:00<?, ?it/s]

CLIP Score: 17.457204818725586
Model: Runway Stable Diffusion
Prompt: A futuristic city with flying cars and neon lights at night
Image Path: model_comparison_outputs/Runway_Stable_Diffusion/image_1.png
CLIP Score: 15.894760131835938

Model: Runway Stable Diffusion
Prompt: A serene beach with crystal clear water and a sunset
Image Path: model_comparison_outputs/Runway_Stable_Diffusion/image_2.png
CLIP Score: 14.662257194519043

Model: Runway Stable Diffusion
Prompt: A hyper-realistic portrait of a smiling old man with wrinkles and gray hair
Image Path: model_comparison_outputs/Runway_Stable_Diffusion/image_3.png
CLIP Score: 18.77044677734375

Model: Dreamlike Photoreal 2.0
Prompt: A futuristic city with flying cars and neon lights at night
Image Path: model_comparison_outputs/Dreamlike_Photoreal_2.0/image_1.png
CLIP Score: 15.42233943939209

Model: Dreamlike Photoreal 2.0
Prompt: A serene beach with crystal clear water and a sunset
Image Path: model_comparison_outputs/Dreamlike_Photore

In [3]:
!pip install torch-fidelity

In [5]:
from torch_fidelity import calculate_metrics

fid_result = calculate_metrics(
    input1="model_comparison_outputs/Runway_Stable_Diffusion",
    input2="model_comparison_outputs/Dreamlike_Photoreal_2.0",
    fid=True,  # Change: Specify 'fid=True' instead of 'metric="fid"'
)
print(f"FID Score: {fid_result['frechet_inception_distance']}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:01<00:00, 81.4MB/s]
Extracting statistics from input 1
Looking for samples non-recursivelty in "model_comparison_outputs/Runway_Stable_Diffusion" with extensions png,jpg,jpeg
Found 3 samples
/usr/local/lib/python3.10/dist-packages/torch_fidelity/datasets.py:16: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  img = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes())).view(height, width, 3)
/usr/local/lib/python3.10/dist-packages/torch/utils

FID Score: 351.77262613020025


Frechet Inception Distance: 351.77262613020025
